# Day 09 — Visualization for Analytics
**Estimated time:** 60–75 min
**Datasets:** `sales_orders.csv`, `materials_inventory.csv`

## Learning Objectives
- Build analytics-grade charts with matplotlib and pandas `.plot()`
- Create a 2x2 KPI dashboard layout using subplot grids
- Apply professional formatting: titles, labels, gridlines, data labels

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

DATA_DIR = "../datasets"

sales = pd.read_csv(f"{DATA_DIR}/sales_orders.csv")
sales["ERDAT"] = pd.to_datetime(sales["ERDAT"], errors="coerce")
sales["NETWR"] = pd.to_numeric(sales["NETWR"], errors="coerce")

inv = pd.read_csv(f"{DATA_DIR}/materials_inventory.csv")
inv["LABST"] = pd.to_numeric(inv["LABST"], errors="coerce")
inv["STPRS"] = pd.to_numeric(inv["STPRS"], errors="coerce")
inv["INV_VALUE"] = inv["LABST"] * inv["STPRS"]

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})
print("Data loaded.")

In [ ]:
# -- 1. Line chart: monthly revenue --
monthly_rev = (sales.set_index("ERDAT")
                    .resample("ME")["NETWR"]
                    .sum()
                    .tail(18))

fig, ax = plt.subplots(figsize=(10, 4))
monthly_rev.plot(ax=ax, marker="o", linewidth=2, color="#2563EB", markersize=5)

ax.set_title("Monthly Revenue — Last 18 Months", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("")
ax.set_ylabel("Revenue ($)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f"${x/1e6:.1f}M" if x >= 1e6 else f"${x:,.0f}"))
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# -- 2. Horizontal bar chart: top 10 customers by revenue --
top10_cust = (sales.groupby("KUNNR")["NETWR"]
                   .sum()
                   .sort_values(ascending=False)
                   .head(10))

fig, ax = plt.subplots(figsize=(8, 5))
top10_cust.sort_values().plot.barh(ax=ax, color="#16A34A", edgecolor="none")

for bar in ax.patches:
    ax.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height() / 2,
            f"${bar.get_width():,.0f}",
            va="center", ha="left", fontsize=9)

ax.set_title("Top 10 Customers by Revenue", fontsize=13, fontweight="bold")
ax.set_xlabel("Total Revenue ($)")
ax.set_ylabel("Customer (KUNNR)")
ax.grid(axis="x", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# -- 3. Donut chart: order status distribution --
status_counts = sales["STATUS"].value_counts()
palette = ["#2563EB","#16A34A","#DC2626","#9333EA","#F59E0B"]

fig, ax = plt.subplots(figsize=(6, 6))
wedges, texts, autotexts = ax.pie(
    status_counts,
    labels=status_counts.index,
    autopct="%1.1f%%",
    startangle=140,
    wedgeprops={"width": 0.6},
    colors=palette[:len(status_counts)]
)
for at in autotexts:
    at.set_fontsize(10)
ax.set_title("Order Status Distribution", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# -- 4. Histogram: inventory value distribution (log scale) --
inv_values = inv["INV_VALUE"].dropna()
inv_values = inv_values[inv_values > 0]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(np.log10(inv_values + 1), bins=30, color="#7C3AED", edgecolor="white", linewidth=0.5)
ax.set_title("Inventory Value Distribution (log10 scale)", fontsize=13, fontweight="bold")
ax.set_xlabel("log10(Inventory Value)")
ax.set_ylabel("Count of Materials")
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
%%time
# -- 5. 2x2 KPI Dashboard --

# Prepare data
monthly_rev_18 = (sales.set_index("ERDAT").resample("ME")["NETWR"].sum().tail(18))
top10 = sales.groupby("KUNNR")["NETWR"].sum().sort_values(ascending=False).head(10)
status_dist = sales["STATUS"].value_counts()
inv_tiers = pd.cut(inv["INV_VALUE"].fillna(0),
                   bins=[-1, 0, 1_000, 10_000, float("inf")],
                   labels=["Zero","< $1K","$1K-$10K","> $10K"])
tier_counts = inv_tiers.value_counts().sort_index()

palette = ["#2563EB","#16A34A","#DC2626","#9333EA","#F59E0B"]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Inventory & Sales KPI Dashboard", fontsize=16, fontweight="bold", y=1.01)

# Panel 1: Revenue by month
ax = axes[0, 0]
monthly_rev_18.plot(ax=ax, color="#2563EB", linewidth=2, marker="o", markersize=4)
ax.set_title("Monthly Revenue (Last 18 Months)", fontweight="bold")
ax.set_ylabel("Revenue ($)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.grid(axis="y", linestyle="--", alpha=0.4)

# Panel 2: Top 10 customers
ax = axes[0, 1]
top10.sort_values().plot.barh(ax=ax, color="#16A34A", legend=False)
ax.set_title("Top 10 Customers by Revenue", fontweight="bold")
ax.set_xlabel("Revenue ($)")
ax.set_ylabel("")
ax.grid(axis="x", linestyle="--", alpha=0.4)

# Panel 3: Order status donut
ax = axes[1, 0]
ax.pie(status_dist, labels=status_dist.index, autopct="%1.1f%%",
       wedgeprops={"width": 0.6}, colors=palette[:len(status_dist)])
ax.set_title("Order Status Distribution", fontweight="bold")

# Panel 4: Inventory value tiers
ax = axes[1, 1]
tier_counts.plot.bar(ax=ax, color="#7C3AED", edgecolor="white", rot=0)
ax.set_title("Materials by Inventory Value Tier", fontweight="bold")
ax.set_ylabel("Count of Materials")
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("day09_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("Dashboard saved to day09_dashboard.png")

---
## Daily Prompt

**Create a 4-panel summary chart:**
1. Revenue by month (line chart, last 12 months)
2. Top 10 customers by revenue (horizontal bar, with data labels)
3. Order status distribution (donut chart)
4. Materials by inventory value tier (bar chart)

The 2x2 dashboard above is a working starting point. Extend it:
- Annotate the highest and lowest revenue months in panel 1
- Add a center label to the donut in panel 3 showing total order count
- Format all monetary values with `$` and thousands separators

```python
# YOUR SOLUTION — improve the dashboard above
# Focus: annotations, cleaner formatting, tighter spacing

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
# YOUR SOLUTION
```

> **Hint:** For month annotations: `ax.annotate("Peak", xy=(top_month, val), xytext=(...))`
> For donut center: `ax.text(0, 0, f"{len(sales):,}\norders", ha="center", va="center", fontsize=12)`